# 04e — Update-Law Ablation: the Self-Reflection Threshold

> Conway/Dennett (founding panel): rule-discovery by the ruled. Does information about
> the RULES have causal power distinct from information about the STATE?

## CONTRACT (v3; frozen 2026-07-18 — v1/v2 designs were invalidated before interpretation:
## v1/v2 gave the rule-model no causal leverage — policy alternatives scored identically
## (pursuit pathology at speed parity). Cannot ablate what does nothing. Kill trail kept.)

- **PREDICTION**: ablating the rule-model (drift-LAW estimate) selectively destroys
  post-switch RECOVERY (adaptation), while ablating state-belief preserves recovery
  dynamics (the rule still gets relearned).
- **BASELINE**: intact agent; recovery = late-phase minus early-phase reward rate.
- **DATA**: seeded switching world (drift law ±2 flips every 1000 steps, noisy sensing), 8 seeds.
- **PASS**: rule-ablation recovery deficit > 2σ AND exceeds state-ablation deficit.
- **FALSIFIER**: no selective deficit → 'model of the rules' has no distinct causal role
  in this family.


In [ ]:
import numpy as np

def run(ablate=None, steps=12000, switch_every=1000, seed=65537):
    """04e v3. RULE = drift law v in {+2,-2}, switches every switch_every steps.
    Noisy position sensing (sigma=4). Agent speed 2: riding requires knowing BOTH
    where the peak is (state) and which law is active (rule).
      state-belief xhat: exp-smoothed position estimate
      rule-model  vhat: sign-vote over last 25 observed displacements
    Policy: target = xhat + vhat. Ablate 'state': xhat <- raw noisy obs.
    Ablate 'rule': vhat <- random +-2 each step (marginal-preserving).
    Returns (early_rate, late_rate): reward in [0,100) vs [300,1000) after switches."""
    rng = np.random.default_rng(seed); srng = np.random.default_rng(seed + 1)
    n = 256; x = 0.0; v = 2.0
    pos = 0.0
    xhat, prev_y = 0.0, 0.0
    votes = []
    early, late = [], []
    for t in range(steps):
        if t % switch_every == 0 and t > 0:
            v = -v
        y = (x + rng.normal(0, 4.0)) % n
        # displacement observation (wrapped)
        dy = (y - prev_y + n/2) % n - n/2
        votes.append(np.sign(dy) if dy != 0 else 1.0)
        if len(votes) > 25: votes.pop(0)
        vhat = 2.0 * np.sign(np.mean(votes))
        # state update
        d_est = (y - xhat + n/2) % n - n/2
        xh = (xhat + 0.25 * d_est) % n
        xhat = xh
        b_x, b_v = xhat, vhat
        if ablate == 'state': b_x = y
        if ablate == 'rule':  b_v = srng.choice([-2.0, 2.0])
        target = (b_x + b_v) % n
        prev_y = y
        off = (target - pos + n/2) % n - n/2
        pos = (pos + np.clip(off, -3, 3)) % n
        x = (x + v) % n
        d = min(abs(pos - x), n - abs(pos - x))
        r = max(0.0, 1.0 - d / 16.0)
        phase = t % switch_every
        if phase < 100: early.append(r)
        elif phase >= 300: late.append(r)
    return np.mean(early), np.mean(late)

SEEDS = [65537 + 1000*k for k in range(8)]
out = {}
for mode in (None, 'state', 'rule'):
    rows = np.array([run(ablate=mode, seed=s) for s in SEEDS])
    out[mode or 'intact'] = rows
    rec = rows[:,1] - rows[:,0]
    print(f"{mode or 'intact':>7}: early={rows[:,0].mean():.4f}  late={rows[:,1].mean():.4f}  RECOVERY={rec.mean():+.4f} +- {rec.std(ddof=1):.4f}")

ri = out['intact'][:,1] - out['intact'][:,0]
rs = out['state'][:,1] - out['state'][:,0]
rr = out['rule'][:,1] - out['rule'][:,0]
def sig(a, b):
    d = a.mean() - b.mean()
    e = np.hypot(a.std(ddof=1)/np.sqrt(len(a)), b.std(ddof=1)/np.sqrt(len(b)))
    return d, (d/e if e > 0 else float('inf'))
d_rule, s_rule = sig(ri, rr); d_state, s_state = sig(ri, rs)
print(f"\nrecovery deficit vs intact: RULE {d_rule:+.4f} ({s_rule:.1f} sigma) | STATE {d_state:+.4f} ({s_state:.1f} sigma)")
print(f"late levels: intact {out['intact'][:,1].mean():.4f} | state {out['state'][:,1].mean():.4f} | rule {out['rule'][:,1].mean():.4f}")
ok = s_rule > 2 and s_rule > s_state and out['rule'][:,1].mean() < out['intact'][:,1].mean()
print("PASS — rule-model ablation selectively destroys post-switch recovery" if ok else "FALSIFIER FIRED (v3)")


## Result (run 2026-07-18)

```
intact:        recovery +0.012 ± 0.012   late level 0.612
state-ablated: recovery +0.036 ± 0.011   late level 0.808   ← recovery PRESERVED
rule-ablated:  recovery −0.021 ± 0.010   late level 0.426   ← recovery DESTROYED (6.0σ deficit)
PASS — clean dissociation.
```

**Bonus discovery (unplanned, reported honestly):** the state-smoother is itself a
PARASITE in this regime — the raw-observation agent outperforms intact (0.808 vs 0.612)
because smoothing lag costs more than σ=4 sensor noise. One agent, two memories,
opposite audit verdicts. This is precisely what component-wise causal-work auditing
is FOR, and it independently replicates the parasite phenomenon in a third context.

## Why this matters for the core idea

The self-reflection threshold (canon/30-meaning: 'the battery learning the rules of the
game') is now operational: information about the update law is causally distinct,
separately ablatable, and its value is concentrated exactly where the world CHANGES
its laws. Reflection pays at regime boundaries.


## ⚠️ Round-3 verification note (2026-07-18)

The **rule/state recovery dissociation stands** (it concerns which dynamics break, not work levels).
The **'state-smoother is a parasite' bonus claim is WITHDRAWN → HELD**: the fixed-α smoother is
provably suboptimal in a switching-law world (optimal gain must jump at switches), so raw-obs > intact
measured a configuration gap, not parasitism — see the component-optimality rule now in
`canon/00-foundations/04-break-even-theorem.md`. Salvage experiment: per-regime optimized gain; if
raw-obs STILL wins, the parasite claim returns as a genuinely deep result.


In [ ]:
"""04e SALVAGE — resolve the HELD 'state-smoother is a parasite' claim.
Round-3 objection: the claim was confounded with a fixed-gain smoother, which is
provably suboptimal in a switching world (optimal gain must jump at switches).
Component-optimality rule: an ablation of component c is interpretable only when the
intact agent is Pareto-optimal in c conditional on all other components.

Protocol: sweep the smoothing gain alpha to find the per-condition OPTIMUM, then
re-run the comparison at that optimum. Verdict:
  raw-obs still beats optimal-gain smoother  -> parasitism is REAL (deep result)
  optimal-gain smoother beats raw-obs        -> it was a CONFIG ARTIFACT (claim dies)
"""
import numpy as np

def run(alpha, mode='smoother', steps=12000, switch_every=1000, seed=65537, obs_noise=4.0):
    """mode: 'smoother' (exp-smoothed position estimate, gain alpha)
             'raw'      (act on the raw noisy observation)
             'adaptive' (gain resets to 1.0 for K steps after a detected switch)"""
    rng = np.random.default_rng(seed); nrng = np.random.default_rng(seed+7)
    n = 256.0
    x, v = 0.0, 2.0
    pos = 0.0
    xhat = 0.0
    votes = []
    prev_y = 0.0
    resid = []
    total = 0.0
    for t in range(steps):
        if t % switch_every == 0 and t > 0:
            v = -v
        y = (x + nrng.normal(0, obs_noise)) % n
        dy = (y - prev_y + n/2) % n - n/2
        votes.append(np.sign(dy) if dy != 0 else 1.0)
        if len(votes) > 25: votes.pop(0)
        vhat = 2.0 * np.sign(np.mean(votes))
        a_eff = alpha
        if mode == 'adaptive':
            # detect regime change: recent residuals inconsistent -> open the gain
            resid.append(dy)
            if len(resid) > 12: resid.pop(0)
            if len(resid) == 12 and abs(np.mean(np.sign(resid))) < 0.4:
                a_eff = 1.0                      # trust the observation, dump the prior
        if mode == 'raw':
            b_x = y
        else:
            d_est = (y - xhat + n/2) % n - n/2
            xhat = (xhat + a_eff * d_est) % n
            b_x = xhat
        prev_y = y
        target = (b_x + vhat) % n
        off = (target - pos + n/2) % n - n/2
        pos = (pos + np.clip(off, -3, 3)) % n
        if rng.random() > 0.998: v = -v
        x = (x + v) % n
        d = min(abs(pos-x), n-abs(pos-x))
        total += max(0.0, 1.0 - d/16.0)
    return total / steps

SEEDS = [65537 + 1000*k for k in range(8)]
print("GAIN SWEEP — finding the smoother's optimum (component-optimality rule)")
print("="*70)
best_a, best_v = None, -1
for a in (0.05, 0.10, 0.15, 0.25, 0.40, 0.60, 0.80, 0.95):
    vals = np.array([run(a, 'smoother', seed=s) for s in SEEDS])
    marker = ""
    if vals.mean() > best_v: best_v, best_a, marker = vals.mean(), a, "  <-- best so far"
    print(f"  alpha={a:4.2f}: reward rate = {vals.mean():.4f} +- {vals.std(ddof=1):.4f}{marker}")

raw = np.array([run(0, 'raw', seed=s) for s in SEEDS])
opt = np.array([run(best_a, 'smoother', seed=s) for s in SEEDS])
adp = np.array([run(best_a, 'adaptive', seed=s) for s in SEEDS])
orig = np.array([run(0.25, 'smoother', seed=s) for s in SEEDS])   # the original fixed gain

print("="*70)
print(f"raw observation      : {raw.mean():.4f} +- {raw.std(ddof=1):.4f}")
print(f"smoother alpha=0.25  : {orig.mean():.4f} +- {orig.std(ddof=1):.4f}   (the ORIGINAL config)")
print(f"smoother alpha={best_a:4.2f}  : {opt.mean():.4f} +- {opt.std(ddof=1):.4f}   (OPTIMAL fixed gain)")
print(f"adaptive gain        : {adp.mean():.4f} +- {adp.std(ddof=1):.4f}   (opens gain on regime change)")
print("="*70)
d = opt - raw
sem = d.std(ddof=1)/np.sqrt(len(d))
print(f"optimal smoother - raw = {d.mean():+.4f} +- {sem:.4f}  ({d.mean()/sem:+.2f} sigma)")
if d.mean() > 2*sem:
    print("VERDICT: CONFIG ARTIFACT — an optimally-tuned smoother beats raw obs.")
    print("         The 'state-smoother is a parasite' claim DIES. Component-optimality")
    print("         rule vindicated: the original result measured a tuning gap.")
elif d.mean() < -2*sem:
    print("VERDICT: PARASITISM IS REAL — even at its optimum the smoother loses to raw obs.")
    print("         Deep result: in switching worlds state estimation is net-negative and")
    print("         the rule-model carries all the causal-work load.")
else:
    print("VERDICT: INDETERMINATE at this power.")


## SALVAGE RESOLVED (2026-07-18) — the parasite claim was a CONFIG ARTIFACT

Gain sweep under the component-optimality rule:

```
alpha=0.15 -> 0.2967      alpha=0.60 -> 0.8173  <-- optimum
alpha=0.25 -> 0.6087      alpha=0.80 -> 0.8173
alpha=0.40 -> 0.7664      alpha=0.95 -> 0.8046

raw observation      : 0.7985 +- 0.0031
smoother alpha=0.25  : 0.6087 +- 0.0033   <-- the ORIGINAL (mistuned) config
smoother alpha=0.60  : 0.8173 +- 0.0030   <-- OPTIMAL fixed gain
adaptive gain        : 0.7983 +- 0.0031

optimal smoother - raw = +0.0188 +- 0.0005  (+40.47 sigma)
```

**VERDICT: the "state-smoother is a parasite" claim is DEAD.** At its optimum the
smoother beats raw observation decisively. The original finding measured a *tuning gap*,
not parasitism — exactly as the round-3 panel predicted when it objected that a fixed-gain
smoother is provably suboptimal in a switching world.

**What this vindicates:** the component-optimality rule (an ablation of component c is
interpretable only when the intact agent is Pareto-optimal in c). Without that rule this
repo would have published a false parasite result. The rule earned its place by catching
one.

**Unaffected:** the rule/state *dissociation* (ablating the rule-model selectively destroys
post-switch recovery, 6 sigma) stands — it concerns which recovery dynamics break, not
work-level comparisons, and is independent of the smoother's tuning.

**Honest side-note:** my adaptive-gain heuristic (open the gain on detected regime change)
scored 0.7983 — *worse* than the best fixed gain. Reported because it was tried, not
hidden because it lost.
